# Data Date-Range Exploration

Quick read-only inspection of the five raw 1-minute `.txt` files.
Checks date coverage per ticker before designing walk-forward
validation periods.

**Not a stage notebook** — no bootstrap, no model training, no artifacts.

In [ ]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("/content/lst_models_raw_stock_data")
RAW_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_FILES = {
    "CSCO": "17A49kUiMELuQqdkOhw1KrpudjP5i5xIN",
    "JPM":  "11UQUJKVXTrBb8XFWY5Z8JDQ8_4i_DE-q",
    "KO":   "1XmtwuZ2dTP20NsU27w5dMyRdSvdnNTSn",
    "MSFT": "1Ud1SQpQbaiRKemFf9dgu1o_raUPnFvGs",
    "WMT":  "1NNfsoUJrrsj2ae5EnC-PTPcZs_QGR_7c",
}

HOLDOUT_BOUNDARY = pd.Timestamp("2017-01-25")

## Download raw files from Google Drive

In [ ]:
import gdown

for ticker, file_id in DRIVE_FILES.items():
    dest = RAW_DIR / f"{ticker}.txt"
    if dest.exists():
        print(f"{ticker}: already present at {dest}")
        continue
    url = f"https://drive.google.com/uc?id={file_id}"
    gdown.download(url, str(dest), quiet=False)
    print(f"{ticker}: downloaded to {dest}")

## Load and inspect each ticker

In [ ]:
COLUMNS = ["Date", "Time", "Open", "High", "Low", "Close", "Volume"]

frames = {}
for ticker in DRIVE_FILES:
    path = RAW_DIR / f"{ticker}.txt"
    df = pd.read_csv(path, names=COLUMNS, parse_dates={"date": ["Date"]},
                     date_format="%m/%d/%Y")
    frames[ticker] = df

for ticker, df in frames.items():
    first, last = df["date"].min(), df["date"].max()
    total = len(df)
    after_holdout = (df["date"] > HOLDOUT_BOUNDARY).sum()

    print(f"\n{'='*50}")
    print(f"{ticker}")
    print(f"{'='*50}")
    print(f"First date : {first.date()}")
    print(f"Last date  : {last.date()}")
    print(f"Total rows : {total:,}")
    print(f"Rows after {HOLDOUT_BOUNDARY.date()}: {after_holdout:,}")
    print(f"\nRows per year:")
    yearly = df.groupby(df["date"].dt.year).size()
    yearly.index.name = "year"
    print(yearly.to_string())

## Summary table across all tickers

In [ ]:
summary_rows = []
for ticker, df in frames.items():
    summary_rows.append({
        "ticker": ticker,
        "min_date": df["date"].min().date(),
        "max_date": df["date"].max().date(),
        "total_rows": len(df),
        "rows_after_2017-01-25": (df["date"] > HOLDOUT_BOUNDARY).sum(),
        "last_year": df["date"].max().year,
    })

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))

## Suggested walk-forward test period boundaries

In [ ]:
global_max = max(df["date"].max() for df in frames.values())
global_min = min(df["date"].min() for df in frames.values())

print(f"Global date range: {global_min.date()} to {global_max.date()}")
print(f"Current holdout boundary: {HOLDOUT_BOUNDARY.date()}")
print()

start = HOLDOUT_BOUNDARY
period = 0
periods_rows = []
print("Suggested ~1-year walk-forward periods from holdout boundary:")
print(f"{'Period':<8} {'Start':>12} {'End':>12}")
print("-" * 34)
while start < global_max:
    end = min(start + pd.DateOffset(years=1), global_max)
    period += 1
    periods_rows.append({"period": period, "start": str(start.date()), "end": str(end.date())})
    print(f"{period:<8} {str(start.date()):>12} {str(end.date()):>12}")
    start = end

periods_df = pd.DataFrame(periods_rows)

print()
print("Note: actual boundaries should align with trading days.")
print("Verify row counts per period before committing to a split.")

## Save results to Google Drive

Saves the summary CSV and walk-forward periods CSV to
`My Drive/lst_models/results/data_date_range_exploration/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/lst_models/results/data_date_range_exploration")
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

summary.to_csv(DRIVE_RESULTS_DIR / "ticker_date_range_summary.csv", index=False)
periods_df.to_csv(DRIVE_RESULTS_DIR / "suggested_walkforward_periods.csv", index=False)

yearly_rows = []
for ticker, df in frames.items():
    counts = df.groupby(df["date"].dt.year).size().reset_index(name="rows")
    counts.columns = ["year", "rows"]
    counts.insert(0, "ticker", ticker)
    yearly_rows.append(counts)
yearly_all = pd.concat(yearly_rows, ignore_index=True)
yearly_all.to_csv(DRIVE_RESULTS_DIR / "rows_per_year_by_ticker.csv", index=False)

print(f"Saved to: {DRIVE_RESULTS_DIR}")
for f in sorted(DRIVE_RESULTS_DIR.glob("*.csv")):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")